# Figure 1b

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# --------------------------------
# Journal / presentation style
# --------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "figure.dpi": 100,
    "savefig.dpi": 400,
    "axes.linewidth": 1.2,
    "font.size": 24,
})

# --------------------------------
# Parameters
# --------------------------------
chi12 = 10.0
res = 55
eps = 1e-3

# --------------------------------
# Free energy
# --------------------------------
def fh_free_energy(w1, w2):
    return (
        0.5 * chi12 * w1 * w2
        + w1 * np.log(w1)
        + w2 * np.log(w2)
        + (1-w1) * np.log(1-w1)
        + (1-w2) * np.log(1-w2)
    )

# --------------------------------
# Grid
# --------------------------------
w1 = np.linspace(eps, 1.0-eps, res)
w2 = np.linspace(eps, 1.0-eps, res)

W1, W2 = np.meshgrid(w1, w2)
F_raw = fh_free_energy(W1, W2)

# --------------------------------
# Energy transform
# --------------------------------
zmin_raw = F_raw.min()

F_plot = np.log(F_raw - zmin_raw + 1.0)
zmin = F_plot.min()
zmax = F_plot.max()
zbase = zmin - 0.2

# --------------------------------
# Constraint line: w1 + w2 = 1
# --------------------------------
line_w1 = np.linspace(eps, 1.0 - eps, 800)
line_w2 = 1.0 - line_w1

line_F_raw = fh_free_energy(line_w1, line_w2)
line_F_plot = np.log(line_F_raw - zmin_raw + 1.0)

# --------------------------------
# Figure
# --------------------------------
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")

# --------------------------------
# Surface with manual shading
# --------------------------------
ls = LightSource(azdeg=230, altdeg=45)

base_color = np.array([232, 163, 75]) / 255.0  # "#E8A34B"

rgb = np.zeros((*F_plot.shape, 3))
rgb[..., 0] = base_color[0]
rgb[..., 1] = base_color[1]
rgb[..., 2] = base_color[2]

shaded_rgb = ls.shade_rgb(
    rgb,
    F_plot,
    vert_exag=1.2,
    blend_mode="soft"
)

facecolors = np.zeros((*F_plot.shape, 4))
facecolors[..., :3] = shaded_rgb
facecolors[..., 3] = 0.90

# Layer 1: shaded surface
ax.plot_surface(
    W1,
    W2,
    F_plot,
    facecolors=facecolors,
    linewidth=0,
    antialiased=True,
    shade=False,
    alpha=0.9,
)

# Layer 2: visible mesh / wireframe
ax.plot_wireframe(
    W1,
    W2,
    F_plot + 1e-4,
    color="#6A4720",
    linewidth=0.45,
    rstride=3,
    cstride=3,
    alpha=0.85,
)

# --------------------------------
# Constraint curve on surface
# --------------------------------
ax.plot(
    line_w1,
    line_w2,
    line_F_plot + 2e-4,
    color="black",
    lw=2.2,
    ls="-",
    solid_capstyle="round",
    zorder=200
)

# --------------------------------
# Base-plane contours
# --------------------------------
custom_levels = [-0.7, -0.5, 0.1, 0.7, 1.5]

ax.contour(
    W1,
    W2,
    F_raw,
    levels=custom_levels,
    zdir="z",
    offset=zbase,
    colors="0.50",
    linewidths=0.9,
    linestyles="--"
)

# --------------------------------
# Projected constraint line on base plane
# --------------------------------
ax.plot(
    line_w1,
    line_w2,
    np.full_like(line_w1, zbase),
    color="black",
    lw=1.4,
    ls="--",
    alpha=0.25,
    zorder=80
)

# --------------------------------
# Projection guide lines from constraint curve
# --------------------------------
n_guides = 5
indices = np.linspace(0, len(line_w1) - 1, n_guides, dtype=int)

for i in indices:
    if np.isfinite(line_F_plot[i]):
        ax.plot(
            [line_w1[i], line_w1[i]],
            [line_w2[i], line_w2[i]],
            [zbase, line_F_plot[i]],
            color="0.35",
            lw=0.8,
            ls="--",
            alpha=0.45,
            zorder=70
        )

# --------------------------------
# Base rectangle
# --------------------------------
ax.plot(
    [eps, 1, 1, eps, eps],
    [eps, eps, 1, 1, eps],
    [zbase] * 5,
    color="black",
    lw=1.8
)

# --------------------------------
# Limits
# --------------------------------
ax.set_xlim(eps, 1.0)
ax.set_ylim(eps, 1.0)
ax.set_zlim(zbase, zmax)

# --------------------------------
# Custom coordinate axes
# --------------------------------
x0 = eps
y0 = eps

ax.plot([x0, 1.00], [y0, y0], [zbase, zbase], color="black", lw=1.8)
ax.plot([x0, x0], [y0, 1.00], [zbase, zbase], color="black", lw=1.8)
ax.plot([x0, x0], [y0, y0], [zbase, zmax], color="black", lw=1.8)

# --------------------------------
# Hide default axes
# --------------------------------
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_zlabel("")

ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.grid(False)

try:
    ax.xaxis.line.set_linewidth(0.0)
    ax.yaxis.line.set_linewidth(0.0)
    ax.zaxis.line.set_linewidth(0.0)

    ax.xaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.yaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.zaxis.pane.set_edgecolor((1, 1, 1, 0))
except Exception:
    pass

# --------------------------------
# View / aspect
# --------------------------------
ax.view_init(elev=32, azim=-65)

try:
    ax.set_box_aspect((1.3, 1.2, 0.8))
except Exception:
    pass

plt.tight_layout()
plt.show()

# Figure 1c

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.colors import LightSource
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# --------------------------------
# Journal / presentation style
# --------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "figure.dpi": 220,
    "savefig.dpi": 600,
    "axes.linewidth": 1.2,
    "font.size": 24,
})

# --------------------------------
# Flory-Huggins parameters
# --------------------------------
N1 = N2 = N3 = 1.0
chi12 = 3.0
chi13 = 3.0
chi23 = 3.0

res = 60
eps = 1e-3

# --------------------------------
# Free energy
# --------------------------------
def fh_free_energy(phi1, phi2, phi3):
    return (
        (phi1 / N1) * np.log(phi1)
        + (phi2 / N2) * np.log(phi2)
        + (phi3 / N3) * np.log(phi3)
        + chi12 * phi1 * phi2
        + chi13 * phi1 * phi3
        + chi23 * phi2 * phi3
    )

# --------------------------------
# Build ternary grid
# --------------------------------
x_list, y_list, f_list = [], [], []

for i in range(res):
    for j in range(res - i):
        phi1 = i / (res - 1)
        phi2 = j / (res - 1)
        phi3 = 1.0 - phi1 - phi2

        if phi1 < eps or phi2 < eps or phi3 < eps:
            continue

        f_val = fh_free_energy(phi1, phi2, phi3)

        x_val = phi2 + 0.5 * phi3
        y_val = (np.sqrt(3) / 2.0) * phi3

        x_list.append(x_val)
        y_list.append(y_val)
        f_list.append(f_val)

x = np.array(x_list)
y = np.array(y_list)
f = np.array(f_list)

tri = mtri.Triangulation(x, y)

# --------------------------------
# Manually keep fewer triangle edges
# --------------------------------
edge_stride = 2

kept_triangles = tri.triangles[::edge_stride]
tri_edges = mtri.Triangulation(x, y, kept_triangles)

# --------------------------------
# Figure
# --------------------------------
fig = plt.figure(figsize=(11.5, 7.8))
ax = fig.add_subplot(111, projection="3d")

zmin = f.min()
zmax = f.max()
zrange = zmax - zmin
zbase = zmin - 0.10 * zrange

# --------------------------------
# Shaded trisurface
# --------------------------------
ls = LightSource(azdeg=0, altdeg=85)

base_color = np.array([232, 163, 75]) / 255.0  # "#E8A34B"

# Use trisurf with shade=True for robust triangular shading
ax.plot_trisurf(
    tri,
    f,
    color="#E8A34B",
    linewidth=0,
    antialiased=True,
    shade=True,
    alpha=0.92
)

# --------------------------------
# Fewer visible triangular mesh edges
# --------------------------------
ax.plot_trisurf(
    tri_edges,
    f,
    color=(0, 0, 0, 0),
    edgecolor="#6A4720",
    linewidth=0.15,
    antialiased=True,
    shade=False,
    alpha=0.00
)

# --------------------------------
# Base triangle
# --------------------------------
verts = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.5, np.sqrt(3) / 2.0],
    [0.0, 0.0]
])

ax.plot(
    verts[:, 0],
    verts[:, 1],
    zs=zbase,
    color="black",
    lw=1.8
)

# --------------------------------
# Subtle base-plane contours
# --------------------------------
ax.tricontour(
    tri,
    f,
    levels=5,
    zdir="z",
    offset=zbase,
    colors="0.50",
    linewidths=0.9
)

# --------------------------------
# Ternary edge tick marks
# --------------------------------
tick_fracs = [0.25, 0.5, 0.75]

for t in tick_fracs:
    xb, yb = t, 0.0
    ax.plot([xb, xb], [yb, yb + 0.018], [zbase, zbase], color="black", lw=1.1)

    x0_left = 0.5 * (1 - t)
    y0_left = (np.sqrt(3) / 2) * (1 - t)
    ax.plot(
        [x0_left, x0_left - 0.014],
        [y0_left, y0_left - 0.010],
        [zbase, zbase],
        color="black",
        lw=1.1
    )

    x0_right = 0.5 + 0.5 * t
    y0_right = (np.sqrt(3) / 2) * (1 - t)
    ax.plot(
        [x0_right, x0_right + 0.014],
        [y0_right, y0_right - 0.010],
        [zbase, zbase],
        color="black",
        lw=1.1
    )

# --------------------------------
# Limits
# --------------------------------
ax.set_xlim(-0.10, 1.03)
ax.set_ylim(-0.07, np.sqrt(3)/2 + 0.08)
ax.set_zlim(zbase, zmax)

# --------------------------------
# Custom minimal coordinate axes
# --------------------------------
x_axis0 = 0.0
y_axis0 = -0.055

ax.plot(
    [x_axis0, 1.02],
    [y_axis0, y_axis0],
    [zbase, zbase],
    color="black",
    lw=1.8
)

ax.plot(
    [x_axis0, x_axis0],
    [y_axis0, np.sqrt(3)/2 + 0.06],
    [zbase, zbase],
    color="black",
    lw=1.8
)

ax.plot(
    [x_axis0, x_axis0],
    [y_axis0, y_axis0],
    [zbase, zmax],
    color="black",
    lw=1.8
)

# --------------------------------
# Hide default axes
# --------------------------------
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_zlabel("")
ax.set_title("Ternary Flory–Huggins free-energy landscape", pad=16, fontsize=19)

ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.grid(False)

try:
    ax.xaxis.line.set_linewidth(0.0)
    ax.yaxis.line.set_linewidth(0.0)
    ax.zaxis.line.set_linewidth(0.0)

    ax.xaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.yaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.zaxis.pane.set_edgecolor((1, 1, 1, 0))
except Exception:
    pass

# --------------------------------
# View / aspect
# --------------------------------
ax.view_init(elev=24, azim=-36)

try:
    ax.set_box_aspect((1.45, 1.18, 0.72))
except Exception:
    pass

plt.tight_layout()
plt.show()

# Figure 1d

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# --------------------------------
# Style: match previous plot
# --------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "figure.dpi": 220,
    "savefig.dpi": 600,
    "axes.linewidth": 1.2,
    "font.size": 15,
})

# --------------------------------
# Conceptual high-dimensional landscape
# 2D projection with four minima
# --------------------------------
x = np.linspace(-3.0, 3.0, 320)
y = np.linspace(-2.6, 2.6, 280)
X, Y = np.meshgrid(x, y)

# Smooth background
Z = 0.09 * (X**2 + 0.9 * Y**2) + 0.005 * X * Y

# Four wells: (cx, cy, amplitude, width)
wells = [
    (-1.7, -0.9, 0.85, 0.62),
    (-0.8,  1.15, 1.20, 0.35),
    ( 0.95, -0.55, 1.55, 0.48),
    ( 1.75,  1.05, 1.15, 0.48),
]

for cx, cy, amp, sig in wells:
    Z -= amp * np.exp(-((X - cx)**2 + (Y - cy)**2) / (2 * sig**2))

# Shift vertically for nicer appearance
Z = Z - Z.min()
Z = Z - 2.2

# --------------------------------
# Helper for exact point values
# --------------------------------
Z_offset_min = Z.min()

def landscape_value(x0, y0):
    val = 0.09 * (x0**2 + 0.9 * y0**2) + 0.005 * x0 * y0

    for cx, cy, amp, sig in wells:
        val -= amp * np.exp(-((x0 - cx)**2 + (y0 - cy)**2) / (2 * sig**2))

    val = val - Z_offset_min - 2.2
    return val

# --------------------------------
# Figure
# --------------------------------
fig = plt.figure(figsize=(11.5, 7.8))
ax = fig.add_subplot(111, projection="3d")

zmin = Z.min()
zmax = Z.max()
zrange = zmax - zmin
zbase = zmin - 0.10 * zrange

# --------------------------------
# Shaded surface + fewer mesh lines
# --------------------------------
ls = LightSource(azdeg=230, altdeg=45)

base_color = np.array([232, 163, 75]) / 255.0  # "#E8A34B"

rgb = np.zeros((*Z.shape, 3))
rgb[..., 0] = base_color[0]
rgb[..., 1] = base_color[1]
rgb[..., 2] = base_color[2]

shaded_rgb = ls.shade_rgb(
    rgb,
    Z,
    vert_exag=1.2,
    blend_mode="soft"
)

facecolors = np.zeros((*Z.shape, 4))
facecolors[..., :3] = shaded_rgb
facecolors[..., 3] = 0.88

# Smooth shaded surface
ax.plot_surface(
    X,
    Y,
    Z,
    facecolors=facecolors,
    linewidth=0,
    antialiased=True,
    shade=False,
    alpha=0.88,
)

# Fewer visible mesh lines
ax.plot_wireframe(
    X,
    Y,
    Z + 1e-4,
    color="#6A4720",
    linewidth=0.45,
    rstride=16,
    cstride=16,
    alpha=0.75,
)

# --------------------------------
# Base plane footprint
# --------------------------------
xmin, xmax = -3.0, 3.0
ymin, ymax = -2.6, 2.6

base_rect_x = [xmin, xmax, xmax, xmin, xmin]
base_rect_y = [ymin, ymin, ymax, ymax, ymin]
base_rect_z = [zbase] * 5

ax.plot(base_rect_x, base_rect_y, base_rect_z, color="black", lw=1.8)

# Base contours
ax.contour(
    X,
    Y,
    Z,
    zdir="z",
    offset=zbase,
    levels=5,
    colors="0.50",
    linewidths=0.9
)

# --------------------------------
# Mark the four minima
# --------------------------------
minima_xy = np.array([
    (-1.7, -0.9),
    (-0.8,  1.15),
    ( 0.95, -0.55),
    ( 1.75,  1.05),
])

labels = [r"$m_1$", r"$m_2$", r"$m_3$", r"$m_4$"]

for (mx, my), lab in zip(minima_xy, labels):
    mz = landscape_value(mx, my)
    # ax.scatter([mx], [my], [mz], s=34, color="black", depthshade=False)
    # ax.text(mx, my, mz - 0.10, lab, ha="center", va="top", fontsize=14)

# --------------------------------
# Limits
# --------------------------------
ax.set_xlim(xmin - 0.25, xmax + 0.15)
ax.set_ylim(ymin - 0.25, ymax + 0.15)
ax.set_zlim(zbase, zmax)

# --------------------------------
# Custom minimal coordinate axes
# --------------------------------
x0 = xmin - 0.18
y0 = ymin - 0.18

ax.plot(
    [x0, xmax],
    [y0, y0],
    [zbase, zbase],
    color="black",
    lw=1.8
)

ax.plot(
    [x0, x0],
    [y0, ymax],
    [zbase, zbase],
    color="black",
    lw=1.8
)

ax.plot(
    [x0, x0],
    [y0, y0],
    [zbase, zmax],
    color="black",
    lw=1.8
)

# --------------------------------
# Hide default axes
# --------------------------------
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])

ax.set_xlabel("")
ax.set_ylabel("")
ax.set_zlabel("")
ax.set_title("Conceptual high-dimensional energy landscape", pad=16, fontsize=19)

ax.xaxis.pane.fill = False
ax.yaxis.pane.fill = False
ax.zaxis.pane.fill = False
ax.grid(False)

try:
    ax.xaxis.line.set_linewidth(0.0)
    ax.yaxis.line.set_linewidth(0.0)
    ax.zaxis.line.set_linewidth(0.0)

    ax.xaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.yaxis.pane.set_edgecolor((1, 1, 1, 0))
    ax.zaxis.pane.set_edgecolor((1, 1, 1, 0))
except Exception:
    pass

# --------------------------------
# View / aspect
# --------------------------------
ax.view_init(elev=24, azim=-36)

try:
    ax.set_box_aspect((1.45, 1.18, 0.72))
except Exception:
    pass

plt.tight_layout()
plt.show()